<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/big_C_cloudflare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neccessary Run

In [ ]:
import subprocess
import sys
from IPython.display import display, HTML

# 1. Setup the 2-line scrolling box UI
display(HTML("""
    <style>
        #scroll_output {
            height: 50px; /* Approximately 2 lines */
            overflow-y: scroll;
            background-color: #1e1e1e;
            color: #00ff00;
            padding: 10px;
            font-family: monospace;
            font-size: 14px;
            border: 1px solid #444;
            display: flex;
            flex-direction: column;
        }
    </style>
    <div id="scroll_output">Starting installation...</div>
"""))

def run_and_scroll(commands):
    """Runs list of commands and streams output to the scroll_output div"""
    for cmd in commands:
        process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            # Escape single quotes for JS and update the div
            escaped_line = line.replace("'", "\\'").strip()
            display(HTML(f"""<script>
                var obj = document.getElementById('scroll_output');
                obj.innerHTML += '<div>' + '{escaped_line}' + '</div>';
                obj.scrollTop = obj.scrollHeight;
            </script>"""), display_id='scroll_update')
        process.wait()

# 2. Updated list of commands (Consolidated and Optimized)
commands_to_run = [
    # 1. Download and install the official Google Chrome stable version
    # "wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -",
    # "sh -c 'echo \"deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main\" >> /etc/apt/sources.list.d/google-chrome.list'",
    "apt-get -y update",
    "apt-get install -y google-chrome-stable",

    # 2. Install all Python dependencies in one step (quiet mode)
    "pip install selenium beautifulsoup4 pandas polars playwright chromedriver-autoinstaller xlsxwriter fastexcel curl_cffi scrapling patchright msgspec browserforge nest_asyncio easyocr -q",

    # 3. Install Playwright and Patchright browsers and their OS dependencies
    "playwright install chromium",
    "playwright install-deps",
    "patchright install chromium",
    "patchright install-deps"
]

run_and_scroll(commands_to_run)
print("\n✅ All installations finished.")


✅ All installations finished.


In [ ]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
import xlsxwriter
import datetime
from datetime import date
import IPython.display as display
import datetime
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

Today is 2026-05-05


In [ ]:
# @title Processing OCR
import io
import re
import requests
import easyocr
import polars as pl
import numpy as np
from PIL import Image

# 1. Initialize OCR
print("🔄 Initializing OCR Engine...")
reader = easyocr.Reader(['en', 'th'])

def get_condition_from_text(raw_text):
    """
    Handles dynamic patterns: Buy 2/3/4/5 Get 1 and Buy 2/3/4 Cheaper.
    Also normalizes Thai keywords and common OCR typos.
    """
    # Normalize text
    t = raw_text.upper().replace(" ", "")
    t = t.replace("BUV", "BUY")  # Fix common OCR typo

    # Extract the first digit found in the text (e.g., '3' from 'BUY3GET1')
    digits = re.findall(r'\d', t)
    n = digits[0] if digits else ""

    # --- 1. Supersave Case ---
    if any(k in t for k in ["SUPERSAVE", "SAVE", "ประหยัด"]):
        return "Supersave"

    # --- 2. Buy N Get 1 Case (Thai & English) ---
    if any(k in t for k in ["GET", "แถม"]):
        if n:
            # Special case for 1 Get 1
            if n == "1" or "1แถม1" in t or "1GET1" in t:
                return "Buy 1 Get 1"
            return f"Buy {n} Get 1"
        return "Buy 1 Get"

    # --- 3. Buy N Cheaper Case ---
    if "CHEAPER" in t:
        if n:
            return f"Buy {n} Cheaper"
        return "Buy 2 Cheaper"

    # Fallback: Return raw text if no pattern matches
    return raw_text.strip() if raw_text.strip() else None



🔄 Initializing OCR Engine...


In [ ]:
# @title udf data-prep
def re_evaluate_price(df: pl.DataFrame) -> pl.DataFrame:
    """
    Standardizes pricing logic:
    1. If original_price is missing, move the promotion_price to it.
    2. If promotion_price matches the original_price, set it to Null.
    """
    return (
        df.with_columns(
            # Step 1: Fix missing original prices by 'swapping' from promotion_price
            pl.when(pl.col("original_price").is_null() & pl.col("promotion_price").is_not_null())
            .then(pl.col("promotion_price"))
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )
        .with_columns(
            # Step 2: Now that original_price is populated, nullify redundant promotions
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("promotion_price"))
            .alias("promotion_price")
        )
    )



In [ ]:
# @title udf Transform (Fixed Volume Extraction)
def parse_product_names(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Pass any supermarket dataframe through this function to standardize the columns.
    Fixed to handle thousands separators (e.g., 1,300 ml).
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Updated patterns
    # Added [\d,.]+ to capture digits, commas, and dots
    # Updated pattern:
    # 1. Added (?)i for case-insensitivity
    # 2. Removed \b to ensure Thai characters don't get blocked by boundary logic
    quant_unit_pattern = r"(?i)([\d.]+)\s*(ML|G|KG|L|GRAMS?)"
    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b)"

    # 3. Apply the Polars transformation
    return df.with_columns(
        pl.lit(today_date).alias("Date"),

        # Extract Brand
        pl.col("name").str.split(" ").list.first().alias("Brand"),

        # Fixed Volume Extraction:
        # 1. Extract the group (e.g., "1,300")
        # 2. Replace commas with nothing so it becomes "1300"
        # 3. Cast to Integer
        pl.col("name")
          .str.extract(quant_unit_pattern, 1)
          .str.replace_all(",", "")
          .cast(pl.Int64, strict=False)
          .alias("Volume"),

        # Extract Unit
        pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),

        # Extract Pack size
        pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Retailer")
    )

# Scrape entire catalog in url list

In [ ]:
%%script echo skipping
# @title work
# This code works
import asyncio
import nest_asyncio
import pandas as pd
from scrapling.fetchers import StealthyFetcher

nest_asyncio.apply()

async def scrape_bigc_multi_pages():
    base_url = "https://www.bigc.co.th/en/category/laundry?brand=184%2C249%2C188%2C189%2C185"
    current_page = 2
    all_data = []

    # กำหนดค่าภาษาอังกฤษ
    en_cookies = [
        {'name': 'language', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'},
        {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'}
    ]
    en_headers = {'Accept-Language': 'en-US,en;q=0.9'}

    print("เริ่มการดึงข้อมูลหลายหน้า...")

    while True:
        target_url = f"{base_url}&page={current_page}"
        print(f"กำลังดึงข้อมูลหน้า {current_page}: {target_url}")

        try:
            page = await StealthyFetcher.async_fetch(
                target_url,
                headless=True,
                solve_cloudflare=True,
                cookies=en_cookies,
                headers=en_headers,
                timeout=60000,
                network_idle=True
            )

            if page.status == 200:
                containers = page.css('div[class*="productCard_container"]')

                # ถ้าไม่พบสินค้าในหน้านี้ แสดงว่าถึงหน้าสุดท้ายแล้ว
                if not containers:
                    print(f"ไม่พบสินค้าเพิ่มเติมที่หน้า {current_page} จบการทำงาน")
                    break

                print(f"พบสินค้า {len(containers)} รายการในหน้า {current_page}")

                for item in containers:
                    name = item.css('div[class*="productCard_title"] a::text').get()
                    sale_price = item.css('span[class*="productCard_sale_price"]::text').get()
                    original_price = item.css('div[class*="productCard_base_price"]::text').get()

                    all_data.append({
                        "page": current_page,
                        "product_name": name.strip() if name else "N/A",
                        "sale_price": sale_price.strip() if sale_price else "N/A",
                        "original_price": original_price.strip().replace('฿', '') if original_price else "N/A"
                    })

                # เพิ่มเลขหน้าเพื่อไปหน้าถัดไป
                current_page += 1

                # ใส่หน่วงเวลาเล็กน้อยเพื่อไม่ให้โดนบล็อก
                await asyncio.sleep(2)
            else:
                print(f"เกิดข้อผิดพลาดที่หน้า {current_page} (Status: {page.status}) หยุดการทำงาน")
                break

        except Exception as e:
            print(f"เกิดข้อผิดพลาดที่หน้า {current_page}: {e}")
            break

    # สรุปผลลัพธ์
    if all_data:
        df = pd.DataFrame(all_data)
        print(f"\nดึงข้อมูลสำเร็จทั้งหมด {len(df)} รายการ จากหน้า 2 ถึง {current_page-1}")
        display(df)
        df.to_csv('bigc_laundry_full.csv', index=False)
        print("บันทึกข้อมูลลงไฟล์ bigc_laundry_full.csv เรียบร้อยแล้ว")
    else:
        print("ไม่พบข้อมูลที่จะบันทึก")

# เริ่มทำงาน
await scrape_bigc_multi_pages()

skipping


In [ ]:
# @title UDF scrape big C polars
# 1. ติดตั้งไลบรารีที่จำเป็น (รวมถึง polars)
# !pip install scrapling patchright msgspec browserforge nest_asyncio polars -q
# !patchright install chromium
# !patchright install-deps

import asyncio
import nest_asyncio
import polars as pl
from scrapling.fetchers import StealthyFetcher

nest_asyncio.apply()

async def get_scrape_bigc_multi_pages_polars(base_url):
    current_page = 1
    all_data = []

    # กำหนดค่าคุกกี้และ headers เพื่อรักษาเซสชันภาษาอังกฤษ
    en_cookies = [
        {'name': 'language', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'},
        {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'}
    ]
    en_headers = {'Accept-Language': 'en-US,en;q=0.9'}

    print("เริ่มการดึงข้อมูลหลายหน้าด้วย Polars...")

    while True:
        # ตรวจสอบโครงสร้าง URL เพื่อต่อพารามิเตอร์ page ให้ถูกต้อง
        separator = "&" if "?" in base_url else "?"
        target_url = f"{base_url}{separator}page={current_page}"
        print(f"กำลังดึงข้อมูลหน้า {current_page}: {target_url}")

        try:
            # ใช้ StealthyFetcher เพื่อข้าม Cloudflare WAF
            page = await StealthyFetcher.async_fetch(
                target_url,
                headless=True,
                solve_cloudflare=True, # จัดการกับระบบตรวจสอบบอทอัตโนมัติ
                cookies=en_cookies,
                headers=en_headers,
                timeout=60000,
                network_idle=True
            )

            if page.status == 200:
                containers = page.css('div[class*="productCard_container"]')

                # หากไม่พบสินค้าในหน้านี้ แสดงว่าถึงหน้าสุดท้ายแล้ว
                if not containers:
                    print(f"ไม่พบสินค้าเพิ่มเติมที่หน้า {current_page} จบการทำงาน")
                    break

                print(f"พบสินค้า {len(containers)} รายการในหน้า {current_page}")

                for item in containers:
                    # สกัดข้อมูลชื่อสินค้าและราคาจาก DOM
                    name = item.css('div[class*="productCard_title"] a::text').get()
                    sale_price = item.css('span[class*="productCard_sale_price"]::text').get()
                    original_price = item.css('div[class*="productCard_base_price"]::text').get()

                    # # Extract the availability status/badge
                    # status = item.css('div[class*="productCard_badge"]::text').get()
                    # if not status:
                    #     status = item.css('div[class*="productCard_label"]::text').get()

                    # # Map "Available soon" or "Out of stock"
                    # is_available = True
                    # if status and ("Soon" in status or "หมด" in status):
                    #     is_available = False

                    # Check for the "Available soon" or "Sold out" badge
                    # Usually found in productCard_badge or productCard_label
                    badge_text = item.css('div[class*="productCard_badge"]::text').get() or ""

                    all_data.append({
                        "page": current_page,
                        "product_name": name.strip() if name else "N/A",
                        "sale_price": sale_price.strip() if sale_price else "N/A",
                        "original_price": original_price.strip().replace('฿', '') if original_price else "N/A",
                        "status": badge_text.strip() if badge_text else "Available"
                    })

                current_page += 1
                # หน่วงเวลาเล็กน้อยเพื่อหลีกเลี่ยงการถูกตรวจจับพฤติกรรม
                await asyncio.sleep(2)
            else:
                print(f"เกิดข้อผิดพลาดที่หน้า {current_page} (Status: {page.status}) หยุดการทำงาน")
                break

        except Exception as e:
            print(f"เกิดข้อผิดพลาดที่หน้า {current_page}: {e}")
            break

    # สรุปผลลัพธ์และส่งคืนค่าเป็น Polars DataFrame
    if all_data:
        df = pl.DataFrame(all_data)
        print(f"\nดึงข้อมูลสำเร็จทั้งหมด {len(df)} รายการ จากหน้า 1 ถึง {current_page-1}")
        return df
    else:
        print("ไม่พบข้อมูลที่จะประมวลผล")
        return pl.DataFrame()

In [ ]:
import asyncio
import nest_asyncio
import polars as pl
from scrapling.fetchers import StealthyFetcher

nest_asyncio.apply()

async def scrape_bigc_multi_pages(base_url_list: list):
    all_data = []

    en_cookies = [
        {'name': 'language', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'},
        {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'}
    ]
    en_headers = {'Accept-Language': 'en-US,en;q=0.9'}

    print(f"🚀 Starting Multi-Category Scrape ({len(base_url_list)} categories)...")

    for base_url in base_url_list:
        current_page = 1
        print(f"\n📂 Processing Category: {base_url}")

        while True:
            separator = "&" if "?" in base_url else "?"
            # Ensure we don't stack multiple page params
            clean_url = base_url.split(f"{separator}page=")[0]
            target_url = f"{clean_url}{separator}page={current_page}"

            print(f"   📄 Page {current_page}...")

            try:
                page = await StealthyFetcher.async_fetch(
                    target_url,
                    headless=True,
                    solve_cloudflare=True,
                    cookies=en_cookies,
                    headers=en_headers,
                    timeout=60000,
                    network_idle=True
                )

                if page.status != 200:
                    print(f"   🛑 Stopped at page {current_page} (Status: {page.status})")
                    break

                containers = page.css('div[class*="productCard_container"]')
                if not containers:
                    print(f"   ✅ Category Complete.")
                    break

                for item in containers:
                    name = item.css('div[class*="productCard_title"] a::text').get()
                    sale_price = item.css('span[class*="productCard_sale_price"]::text').get()
                    original_price = item.css('div[class*="productCard_base_price"]::text').get()
                    badge_url = item.css('div[class*="productCard_badge"] img::attr(src)').get()

                    all_data.append({
                        "product_name": name.strip() if name else "N/A",
                        "sale_price": sale_price.strip() if sale_price else "N/A",
                        "original_price": original_price.strip().replace('฿', '') if original_price else "N/A",
                        "condition": None, # Placeholder for Part 2
                        "badge_url": badge_url if badge_url else "null"
                    })

                current_page += 1
                await asyncio.sleep(1) # Be polite to the server

            except Exception as e:
                print(f"   ❌ Error at page {current_page}: {e}")
                break

    return pl.DataFrame(all_data) if all_data else pl.DataFrame()

# --- RUN SCRAPER ---
category_list = [
    "https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100",
    "https://www.bigc.co.th/en/category/household-dishwashing?limit=100"
]

final_df = await scrape_bigc_multi_pages(category_list)
print(f"🏁 Scrape finished. Total products: {len(final_df)}")

🚀 Starting Multi-Category Scrape (2 categories)...

📂 Processing Category: https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100
   📄 Page 1...


[2026-05-05 06:31:21] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-05-05 06:31:22] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=1> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=1> (referer: https://www.google.com/)


   📄 Page 2...


[2026-05-05 06:32:35] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-05-05 06:33:35] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=2> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=2> (referer: https://www.google.com/)


   📄 Page 3...


...

   📄 Page 2...


[2026-05-05 06:40:29] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-05-05 06:41:30] INFO: Fetched (307) <GET https://www.bigc.co.th/en/category/household-dishwashing?limit=100&page=2> (referer: https://www.google.com/)
INFO:scrapling:Fetched (307) <GET https://www.bigc.co.th/en/category/household-dishwashing?limit=100&page=2> (referer: https://www.google.com/)
[2026-05-05 06:41:30] INFO: Fetched (200) <GET https://www.bigc.co.th/> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/> (referer: https://www.google.com/)


   📄 Page 3...


[2026-05-05 06:41:37] ERROR: No Cloudflare challenge found.
ERROR:scrapling:No Cloudflare challenge found.
[2026-05-05 06:41:37] INFO: Fetched (307) <GET https://www.bigc.co.th/en/category/household-dishwashing?limit=100&page=3> (referer: https://www.google.com/)
INFO:scrapling:Fetched (307) <GET https://www.bigc.co.th/en/category/household-dishwashing?limit=100&page=3> (referer: https://www.google.com/)
[2026-05-05 06:41:38] INFO: Fetched (200) <GET https://www.bigc.co.th/> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/> (referer: https://www.google.com/)


   ✅ Category Complete.
🏁 Scrape finished. Total products: 369


In [ ]:
# --- START PROCESSING ---

# 2. Get unique badge URLs (Excluding nulls)
unique_urls = [url for url in final_df["badge_url"].unique().to_list() if url and url != "null"]

if not unique_urls:
    print("⚠️ No valid badge URLs found.")
else:
    print(f"🔍 Processing {len(unique_urls)} unique badges...")
    badge_map = {}

    for url in unique_urls:
        try:
            headers = {"User-Agent": "Mozilla/5.0"}
            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code == 200:
                img = Image.open(io.BytesIO(response.content)).convert('RGB')

                # Upscale for OCR accuracy
                img = img.resize((img.width * 4, img.height * 4), resample=Image.LANCZOS)

                # Convert to Numpy Array for EasyOCR
                img_array = np.array(img)

                # Run OCR
                results = reader.readtext(img_array)
                raw_text = " ".join([res[1] for res in results])

                # Process text through our new dynamic function
                label = get_condition_from_text(raw_text)
                badge_map[url] = label
                print(f"✅ {url[-15:]} -> {label}")
            else:
                badge_map[url] = None
        except Exception as e:
            print(f"❌ Error on {url[-15:]}: {e}")
            badge_map[url] = None

    # 3. Apply the results back to the 'condition' column
    final_df = final_df.with_columns(
        pl.col("badge_url").replace(badge_map).alias("condition")
    )

    print("\n✨ Process Complete!")
    # Show results where a condition was found
    print(final_df.filter(pl.col("condition").is_not_null()).head(20))

In [ ]:
final_df.to_pandas()

,product_name,sale_price,original_price,condition,badge_url
0,"FINELINE Fabric Softener Gentle White 1,300 ml.",35.00,70.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
1,FINELINE Fabric Softener Red Romance 1300 ml.,35.00,70.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
2,FINELINE Fabric Softener Pink Blossom 1300 ml.,35.00,70.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
3,HYGIENE Expert Wash Concentrate Liquid Deterge...,55.00,65.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
4,FINELINE Plus Liquid Laundry Detergent Sunny G...,179.00,N/A,null,null
...,...,...,...,...,...
364,CRYSTAL Drinking Water 1500 ml. Pack 8,62.00,N/A,ซื้อ2 ถูกลง,https://st.bigc-cs.com/cdn-cgi/image/format=we...
365,BENJARONG RICE Jasmine rice 100% 5 kg,195.00,198.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
366,HALE'S BLUE BOY BRAND Concentrated Sala Flavor...,69.00,N/A,null,null
367,LIN White Sugar 1 kg.,26.00,N/A,null,null


In [ ]:
# final_df.write_excel(f"big_c_{today_date}.xlsx")

In [ ]:
df_big_c = final_df.clone()

In [ ]:
df_big_c_sel = df_big_c.select([
    pl.col("product_name").alias("name"),
    # strict=False จะเปลี่ยนทุกอย่างที่แปลงเป็น Float ไม่ได้ให้เป็น Null
    pl.col("sale_price").cast(pl.Float64, strict=False).alias("promotion_price"),
    pl.col("original_price").cast(pl.Float64, strict=False).alias("original_price"),
    pl.col("condition")
])

In [ ]:
# How to use it:
df_prep_big_c = re_evaluate_price(df_big_c_sel)
df_trans_big_c = parse_product_names(df_prep_big_c, "BigC")

In [ ]:
df_trans_big_c

name,promotion_price,original_price,condition,Date,Brand,Volume,Unit,Pack,Retailer
str,f64,f64,str,str,str,i64,str,str,str
"""FINELINE Fabric Softener Gentl…",35.0,70.0,"""Supersave""","""2026-05-05""","""FINELINE""",1300,"""ML""",null,"""BigC"""
"""FINELINE Fabric Softener Red R…",35.0,70.0,"""Supersave""","""2026-05-05""","""FINELINE""",1300,"""ML""",null,"""BigC"""
"""FINELINE Fabric Softener Pink …",35.0,70.0,"""Supersave""","""2026-05-05""","""FINELINE""",1300,"""ML""",null,"""BigC"""
"""HYGIENE Expert Wash Concentrat…",55.0,65.0,"""Supersave""","""2026-05-05""","""HYGIENE""",600,"""ML""",null,"""BigC"""
"""FINELINE Plus Liquid Laundry D…",null,179.0,"""null""","""2026-05-05""","""FINELINE""",1250,"""ML""",null,"""BigC"""
…,…,…,…,…,…,…,…,…,…
"""CRYSTAL Drinking Water 1500 ml…",null,62.0,"""ซื้อ2 ถูกลง""","""2026-05-05""","""CRYSTAL""",1500,"""ML""","""PACK 8""","""BigC"""
"""BENJARONG RICE Jasmine rice 10…",195.0,198.0,"""Supersave""","""2026-05-05""","""BENJARONG""",5,"""KG""",null,"""BigC"""
"""HALE'S BLUE BOY BRAND Concentr…",null,69.0,"""null""","""2026-05-05""","""HALE'S""",710,"""ML""",null,"""BigC"""


In [ ]:
df_trans_big_c.write_excel(f"big_c_result_{today_date}.xlsx")

# Find watchlist

In [ ]:
# @title udf Scrape Watchlist (Patchright + Isolated Contexts)
import asyncio
import polars as pl
from bs4 import BeautifulSoup
from patchright.async_api import async_playwright
import random

async def scrape_bigc_watchlist_sequential(urls: list):
    final_data = []
    retry_queue = []

    async with async_playwright() as p:
        # Launch the core browser once
        browser = await p.chromium.launch(
            headless=True,
            args=["--disable-blink-features=AutomationControlled"]
        )

        async def process_url(url):
            # 1. Guard against the missing comma bug in Python lists
            if url.count("https://") > 1:
                print(f"   ❌ SYNTAX ERROR: You forgot a comma in your URL list! Two links merged.")
                return None, False

            short_url_name = url.split('/')[-1][:40]
            print(f"🚀 Processing: {short_url_name}...")

            # 2. CREATE A COMPLETELY ISOLATED INCOGNITO CONTEXT FOR EVERY REQUEST
            context = await browser.new_context(
                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
                viewport={"width": 1920, "height": 1080}
            )

            # Inject English cookies into this specific clean context
            await context.add_cookies([
                {'name': 'language', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'},
                {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'}
            ])

            page = await context.new_page()

            try:
                await page.goto(url, wait_until="domcontentloaded", timeout=60000)

                try:
                    await page.wait_for_selector("h1", timeout=10000)
                except:
                    pass

                await asyncio.sleep(2)

                html_content = await page.content()
                soup = BeautifulSoup(html_content, "html.parser")

                name_elem = soup.find("h1")
                name = name_elem.text.strip() if name_elem else "Unknown Product"

                # Better Debugging for Blockers
                if "Just a moment" in name or "Cloudflare" in html_content:
                    print(f"   🛑 Blocked by Cloudflare.")
                    return None, False
                if name == "Unknown Product":
                    print(f"   ⚠️ Page loaded, but couldn't find the product title. May be out of stock/removed.")
                    return None, False

                promo_price = None
                orig_price = None

                price_div = soup.find("div", class_=lambda x: x and "productDetail_product_price_new" in x)

                if price_div:
                    base_div = price_div.find("div", class_=lambda x: x and "productDetail_product_baseprice" in x)
                    if base_div:
                        orig_price = base_div.text.replace('฿', '').replace(',', '').strip()
                        base_div.extract()

                    unit_span = price_div.find("span", class_=lambda x: x and "productDetail_unit" in x)
                    if unit_span:
                        unit_span.extract()

                    promo_price = price_div.text.replace('฿', '').replace(',', '').strip()

                badge_url = "null"
                badge_div = soup.find("div", class_=lambda x: x and "imageSlider_badge_warpper" in x)
                if badge_div:
                    badge_img = badge_div.find("img")
                    if badge_img and badge_img.has_attr('src'):
                        badge_url = badge_img['src']

                return {
                    "url": url,
                    "name": name,
                    "promotion_price": promo_price,
                    "original_price": orig_price,
                    "condition": None,
                    "badge_url": badge_url
                }, True

            except Exception as e:
                print(f"❌ Error on {short_url_name}: {e}")
                return None, False

            finally:
                # 3. DESTROY THE ENTIRE CONTEXT AFTER THE REQUEST
                await context.close()

        print("--- Starting First Pass ---")
        for url in urls:
            data, success = await process_url(url)
            if success:
                print(f"   ✅ Success: ฿{data['promotion_price']} | {data['name'][:30]}...")
                final_data.append(data)
            else:
                retry_queue.append(url)

            # Randomize delay between 3 and 6 seconds to mimic human reading time
            await asyncio.sleep(random.uniform(3.0, 6.0))

        if retry_queue:
            print(f"\n--- Starting Retry Queue ({len(retry_queue)} items) ---")
            await asyncio.sleep(10) # Let the IP cool down
            for url in retry_queue:
                data, success = await process_url(url)
                if success:
                    print(f"   ✅ Success (Retry): ฿{data['promotion_price']} | {data['name'][:30]}...")
                    final_data.append(data)
                else:
                    print(f"   ❌ Permanently failed on retry.")
                    final_data.append({
                        "url": url,
                        "name": "Blocked or Not Found",
                        "promotion_price": None,
                        "original_price": None,
                        "condition": None,
                        "badge_url": "null"
                    })
                await asyncio.sleep(random.uniform(3.0, 6.0))

        await browser.close()

    df = pl.DataFrame(final_data)
    if not df.is_empty():
        df = df.with_columns([
            pl.col("url").cast(pl.String),
            pl.col("name").cast(pl.String),
            pl.col("promotion_price").cast(pl.Float64, strict=False),
            pl.col("original_price").cast(pl.Float64, strict=False),
            pl.col("condition").cast(pl.String, strict=False),
            pl.col("badge_url").cast(pl.String)
        ])

    return df

In [ ]:
# @title RUN Watchlist Scraper
watchlist_urls = [
# -- BIG C
# 'FINELINE Liquid Laundry Detergent Sunny Gold Scent 550 ml.',
"https://www.bigc.co.th/en/product/fineline-liquid-laundry-detergent-sunny-gold-scent-550-ml.3791984",
# 'FINELINE Plus Liquid Laundry Detergent Sunny Gold Scent 1250 ml.',
"https://www.bigc.co.th/en/product/fineline-plus-liquid-laundry-detergent-sunny-gold-scent-1250-ml.2155497",
# 'HYGIENE Expert Wash Concentrate Liquid Detergent Milky Touch 600 ml.',
"https://www.bigc.co.th/en/product/hygiene-expert-wash-concentrate-liquid-detergent-milky-touch-600-ml.6394917",
# 'HYGIENE Expert Wash Concentrate Liquid Detergent Milky Touch 1400 ml.',
"https://www.bigc.co.th/en/product/hygiene-expert-wash-concentrate-liquid-detergent-milky-touch-1400-ml.6394919",
# 'PAO Win Wash Liquid Laundry Detergent 620 ml.',
"https://www.bigc.co.th/en/product/pao-win-wash-liquid-detergent-620-ml.12782",
# 'PAO Win Wash Liquid Laundry Detergent 1300 ml.',
"https://www.bigc.co.th/en/product/pao-win-wash-concentrated-liquid-detergent-formula-1300-ml.34065",
# 'PAO Super White Laundry Detergent 1800 g.',
"https://www.bigc.co.th/en/product/pao-super-white-laundry-detergent-1800-g.5977",
# 'PAO Super White Powder Laundry Detergent 2400 g.',
"https://www.bigc.co.th/en/product/pao-super-white-detergent-2400-g.3520",
# 'ATTACK EASY DETERGENT HAPPY SWEET 2500 G',
"https://www.bigc.co.th/en/product/attack-easy-detergent-happy-sweet-2500-g.47463",
# 'HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 480 ml.',
"https://www.bigc.co.th/en/product/hygiene-fabric-softener-expert-care-milky-touch-480-ml.12003",
# 'HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 1000 ml. Pack 2',
"https://www.bigc.co.th/en/product/hygiene-expert-care-concentrated-fabric-softener-milky-touch-scent-1000-ml-pack-2.1953035",
# 'LIPON F Dishwashing Liquid Hygienic Formula 500 ml. Pack 3',
"https://www.bigc.co.th/en/product/lipon-f-dishwashing-liquid-hygienic-formula-500-ml-pack-3.3639",
# 'LIPON F Dishwashing Liquid Hygienic Formula 3200 ml.',
"https://www.bigc.co.th/en/product/lipon-f-dishwashing-liquid-hygienic-formula-3200-ml.673",
# 'PRO Blue Plus Powder Laundry Detergent 2400 g.',
"https://www.bigc.co.th/en/product/pro-blue-plus-powder-laundry-detergent-standard-formula-2400-g.501",
# 'LIPON F Sanitary Formula Dish Washing Liquid Refill 750 ml. Pack of 2',
"https://www.bigc.co.th/en/product/lipon-f-sanitary-formula-dish-washing-liquid-refill-750-ml-pack-of-2.78902",
# 'HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 480 ml. Pack 2+1',
"https://www.bigc.co.th/en/product/hygiene-fabric-softener-expert-care-tender-touch-480-ml-pack-2-free-1.32428",
# 'ATTACK Easy Conventional Detergent Happy Sweet Pink 1.7 kg x 1+1',
# NOT FOUND
# ATTACK EZ Conventional Detergent Happy Sweet Scent 1700 g.
"https://www.bigc.co.th/en/product/attack-ez-conventional-detergent-happy-sweet-scent-1700-g.47863",
]

# Run the sequential queue version
df_watchlist_final = await scrape_bigc_watchlist_sequential(watchlist_urls)

print("\n--- Watchlist Scrape Complete ---")
display.display(df_watchlist_final)

--- Starting First Pass ---
🚀 Processing: fineline-liquid-laundry-detergent-sunny-...
   ✅ Success: ฿69.00 | FINELINE Liquid Laundry Deterg...
🚀 Processing: fineline-plus-liquid-laundry-detergent-s...
   ✅ Success: ฿179.00 | FINELINE Plus Liquid Laundry D...
🚀 Processing: hygiene-expert-wash-concentrate-liquid-d...
   ✅ Success: ฿55.00 | HYGIENE Expert Wash Concentrat...
🚀 Processing: hygiene-expert-wash-concentrate-liquid-d...
   ✅ Success: ฿114.00 | HYGIENE Expert Wash Concentrat...
🚀 Processing: pao-win-wash-liquid-detergent-620-ml.127...
   ✅ Success: ฿99.00 | PAO Win Wash Liquid Laundry De...
🚀 Processing: pao-win-wash-concentrated-liquid-deterge...
   ✅ Success: ฿185.00 | PAO Win Wash Liquid Laundry De...
🚀 Processing: pao-super-white-laundry-detergent-1800-g...
   ✅ Success: ฿75.00 | PAO Super White Laundry Deterg...
🚀 Processing: pao-super-white-detergent-2400-g.3520...
   ✅ Success: ฿155.00 | PAO Super White Powder Laundry...
🚀 Processing: attack-easy-detergent-happy-sweet-250

url,name,promotion_price,original_price,condition,badge_url
str,str,f64,f64,str,str
"""https://www.bigc.co.th/en/prod…","""FINELINE Liquid Laundry Deterg…",69.0,89.0,null,"""https://st.bigc-cs.com/cdn-cgi…"
"""https://www.bigc.co.th/en/prod…","""FINELINE Plus Liquid Laundry D…",179.0,null,null,"""null"""
"""https://www.bigc.co.th/en/prod…","""HYGIENE Expert Wash Concentrat…",55.0,65.0,null,"""https://st.bigc-cs.com/cdn-cgi…"
"""https://www.bigc.co.th/en/prod…","""HYGIENE Expert Wash Concentrat…",114.0,139.0,null,"""https://st.bigc-cs.com/cdn-cgi…"
"""https://www.bigc.co.th/en/prod…","""PAO Win Wash Liquid Laundry De…",99.0,null,null,"""null"""
…,…,…,…,…,…
"""https://www.bigc.co.th/en/prod…","""LIPON F Dishwashing Liquid Hyg…",118.0,165.0,null,"""https://st.bigc-cs.com/cdn-cgi…"
"""https://www.bigc.co.th/en/prod…","""PRO Blue Plus Powder Laundry D…",99.0,150.0,null,"""https://st.bigc-cs.com/cdn-cgi…"
"""https://www.bigc.co.th/en/prod…","""LIPON F Sanitary Formula Dish …",69.0,85.0,null,"""https://st.bigc-cs.com/cdn-cgi…"


In [ ]:
# @title Run OCR on Watchlist Badges
import io
import requests
import easyocr
import polars as pl
import numpy as np
from PIL import Image
import IPython.display as display

# 1. Initialize OCR (if not already done)
print("🔄 Initializing OCR Engine...")
reader = easyocr.Reader(['en', 'th'])

# 2. Get unique badge URLs from your dataframe (excluding null strings)
unique_urls = [
    url for url in df_watchlist_final["badge_url"].unique().to_list()
    if url and url != "null"
]

if not unique_urls:
    print("⚠️ No valid badge URLs found to process.")
else:
    print(f"🔍 Processing {len(unique_urls)} unique badges...")
    badge_map = {}

    for url in unique_urls:
        try:
            # Download the image
            headers = {"User-Agent": "Mozilla/5.0"}
            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code == 200:
                img = Image.open(io.BytesIO(response.content)).convert('RGB')

                # Upscale by 4x for better OCR accuracy on small badges
                img = img.resize((img.width * 4, img.height * 4), resample=Image.LANCZOS)
                img_array = np.array(img)

                # Read text
                results = reader.readtext(img_array)
                raw_text = " ".join([res[1] for res in results])

                # Route through your custom condition function!
                label = get_condition_from_text(raw_text)
                badge_map[url] = label
                print(f"✅ Extracted: '{raw_text}' -> Mapped to: {label}")
            else:
                badge_map[url] = None

        except Exception as e:
            print(f"❌ Error processing badge: {e}")
            badge_map[url] = None

    # Handle null mapping safely
    badge_map["null"] = None
    badge_map[None] = None

    # 3. Overwrite the condition column in df_watchlist_final
    df_watchlist_final = df_watchlist_final.with_columns(
        pl.col("badge_url").replace(badge_map, default=None).alias("condition")
    )

    print("\n✨ OCR Process Complete!")

    # Display the updated rows
    display.display(
        df_watchlist_final
        .select(["name", "promotion_price", "badge_url", "condition"])
        .filter(pl.col("condition").is_not_null())
    )

🔄 Initializing OCR Engine...
🔍 Processing 2 unique badges...
✅ Extracted: 'buv 1. get' -> Mapped to: Buy 1 Get 1
✅ Extracted: 'ถูกจริง ประหยัดจริง' -> Mapped to: Supersave

✨ OCR Process Complete!


/tmp/ipykernel_6364/3421712054.py:60: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("badge_url").replace(badge_map, default=None).alias("condition")


name,promotion_price,badge_url,condition
str,f64,str,str
"""FINELINE Liquid Laundry Deterg…",69.0,"""https://st.bigc-cs.com/cdn-cgi…","""Supersave"""
"""HYGIENE Expert Wash Concentrat…",55.0,"""https://st.bigc-cs.com/cdn-cgi…","""Supersave"""
"""HYGIENE Expert Wash Concentrat…",114.0,"""https://st.bigc-cs.com/cdn-cgi…","""Supersave"""
"""PAO Win Wash Liquid Laundry De…",185.0,"""https://st.bigc-cs.com/cdn-cgi…","""Buy 1 Get 1"""
"""PAO Super White Laundry Deterg…",75.0,"""https://st.bigc-cs.com/cdn-cgi…","""Supersave"""
…,…,…,…
"""HYGIENE Expert Care Concentrat…",195.0,"""https://st.bigc-cs.com/cdn-cgi…","""Supersave"""
"""LIPON F Dishwashing Liquid Hyg…",69.0,"""https://st.bigc-cs.com/cdn-cgi…","""Supersave"""
"""LIPON F Dishwashing Liquid Hyg…",118.0,"""https://st.bigc-cs.com/cdn-cgi…","""Supersave"""


In [ ]:
df_watchlist_final.to_pandas()

,url,name,promotion_price,original_price,condition,badge_url
0,https://www.bigc.co.th/en/product/fineline-liq...,FINELINE Liquid Laundry Detergent Sunny Gold S...,69.0,89.0,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
1,https://www.bigc.co.th/en/product/fineline-plu...,FINELINE Plus Liquid Laundry Detergent Sunny G...,179.0,NaN,None,null
2,https://www.bigc.co.th/en/product/hygiene-expe...,HYGIENE Expert Wash Concentrate Liquid Deterge...,55.0,65.0,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
3,https://www.bigc.co.th/en/product/hygiene-expe...,HYGIENE Expert Wash Concentrate Liquid Deterge...,114.0,139.0,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
4,https://www.bigc.co.th/en/product/pao-win-wash...,PAO Win Wash Liquid Laundry Detergent 620 ml.,99.0,NaN,None,null
5,https://www.bigc.co.th/en/product/pao-win-wash...,PAO Win Wash Liquid Laundry Detergent 1300 ml.,185.0,NaN,Buy 1 Get 1,https://st.bigc-cs.com/cdn-cgi/image/format=we...
6,https://www.bigc.co.th/en/product/pao-super-wh...,PAO Super White Laundry Detergent 1800 g.,75.0,112.0,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
7,https://www.bigc.co.th/en/product/pao-super-wh...,PAO Super White Powder Laundry Detergent 2400 g.,155.0,NaN,None,null
8,https://www.bigc.co.th/en/product/attack-easy-...,ATTACK EASY DETERGENT HAPPY SWEET 2500 G,159.0,NaN,None,null
9,https://www.bigc.co.th/en/product/hygiene-fabr...,HYGIENE Expert Care Concentrated Fabric Soften...,49.0,60.0,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...


In [ ]:
list_to_search = [
# -- BIG C
'FINELINE Liquid Laundry Detergent Sunny Gold Scent 550 ml.',
'FINELINE Plus Liquid Laundry Detergent Sunny Gold Scent 1250 ml.',
'HYGIENE Expert Wash Concentrate Liquid Detergent Milky Touch 600 ml.',
'HYGIENE Expert Wash Concentrate Liquid Detergent Milky Touch 1400 ml.',
'PAO Win Wash Liquid Laundry Detergent 620 ml.',
'PAO Win Wash Liquid Laundry Detergent 1300 ml.',
'PAO Super White Laundry Detergent 1800 g.',
'PAO Super White Powder Laundry Detergent 2400 g.',
'ATTACK EASY DETERGENT HAPPY SWEET 2500 G',
'HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 480 ml.',
'HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 1000 ml. Pack 2',
'LIPON F Dishwashing Liquid Hygienic Formula 500 ml. Pack 3',
'LIPON F Dishwashing Liquid Hygienic Formula 3200 ml.',
'PRO Blue Plus Powder Laundry Detergent 2400 g.',
'LIPON F Sanitary Formula Dish Washing Liquid Refill 750 ml. Pack of 2',
'HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 480 ml. Pack 2+1',
'ATTACK Easy Conventional Detergent Happy Sweet Pink 1.7 kg x 1+1',
# Found this, above (Not Found)
'ATTACK EZ Conventional Detergent Happy Sweet Scent 1700 g.'
]

search_df = pl.DataFrame({"product_name": list_to_search})

# Join with df_trans_lotuss to get original_price and promotion_price
search_results_df = search_df.join(
    df_watchlist_final.select(["name", "original_price", "promotion_price"]),
    left_on="product_name",
    right_on="name",
    how="left"
)
lotuss_names_set = set(df_watchlist_final["name"].to_list())

search_results_df = search_results_df.with_columns(
    pl.col("product_name").is_in(lotuss_names_set).alias("Found")
).unique()

print("Search Results with Prices:")
print(search_results_df)

Search Results with Prices:
shape: (18, 4)
┌─────────────────────────────────┬────────────────┬─────────────────┬───────┐
│ product_name                    ┆ original_price ┆ promotion_price ┆ Found │
│ ---                             ┆ ---            ┆ ---             ┆ ---   │
│ str                             ┆ f64            ┆ f64             ┆ bool  │
╞═════════════════════════════════╪════════════════╪═════════════════╪═══════╡
│ PAO Super White Laundry Deterg… ┆ 112.0          ┆ 75.0            ┆ true  │
│ HYGIENE Expert Care Concentrat… ┆ 219.0          ┆ 195.0           ┆ true  │
│ HYGIENE Expert Wash Concentrat… ┆ 139.0          ┆ 114.0           ┆ true  │
│ ATTACK Easy Conventional Deter… ┆ null           ┆ null            ┆ false │
│ PAO Super White Powder Laundry… ┆ null           ┆ 155.0           ┆ true  │
│ …                               ┆ …              ┆ …               ┆ …     │
│ HYGIENE Expert Care Concentrat… ┆ 60.0           ┆ 49.0            ┆ true  │
│ ATTACK 

In [ ]:
df_watchlist_final= df_watchlist_final.select([
    pl.col("name"),
    # strict=False จะเปลี่ยนทุกอย่างที่แปลงเป็น Float ไม่ได้ให้เป็น Null
    pl.col("promotion_price").cast(pl.Float64, strict=False),
    pl.col("original_price").cast(pl.Float64, strict=False),
    pl.col("condition")
])
df_prep_big_c_watchlist = re_evaluate_price(df_watchlist_final)
df_trans_big_c_watchlist = parse_product_names(df_prep_big_c_watchlist, "BigC")
df_trans_big_c_watchlist.write_excel(f"big_c_watchlist_{today_date}.xlsx")